In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

# WILL RG Universal Invariants
c = 299792458.0  # m/s
G = 6.6743e-11   # m^3/kg/s^2
M_sun = 1.98847e30 # kg
kpc_to_m = 3.0857e19
Mpc_to_m = 3.0857e22

H0_km_s_Mpc = 68.15
H0_s = (H0_km_s_Mpc * 1000) / Mpc_to_m
R_H = c / H0_s
a_k = (c * H0_s) / (3 * np.pi)

# Data URLs
url_table1 = "https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/DATA/Little_Things/table1.txt"
url_table2 = "https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/DATA/Little_Things/table2.txt"

# Explicit Byte-by-Byte mapping (0-indexed, half-open intervals)
t1_specs = [(0, 8), (32, 36)]
t1_names = ['Name', 'Dist']

t2_specs = [(0, 8), (9, 13), (19, 24), (139, 145), (152, 157)]
t2_names = ['Name', 'Rmax_kpc', 'V_obs_max_km_s', 'Mgas', 'MstarSED']

# Load Data with exact colspecs
table1 = pd.read_fwf(url_table1, colspecs=t1_specs, names=t1_names, header=None)
table2 = pd.read_fwf(url_table2, colspecs=t2_specs, names=t2_names, header=None)

# Clean data (convert to numeric, coercing any whitespace to NaN)
table1['Dist'] = pd.to_numeric(table1['Dist'], errors='coerce')
table2['Rmax_kpc'] = pd.to_numeric(table2['Rmax_kpc'], errors='coerce')
table2['V_obs_max_km_s'] = pd.to_numeric(table2['V_obs_max_km_s'], errors='coerce')
table2['Mgas'] = pd.to_numeric(table2['Mgas'], errors='coerce')
table2['MstarSED'] = pd.to_numeric(table2['MstarSED'], errors='coerce')

# Merge and sanitize
galaxies = pd.merge(table1, table2, on='Name')
galaxies['MstarSED'] = galaxies['MstarSED'].fillna(0)
galaxies = galaxies.dropna(subset=['V_obs_max_km_s', 'Rmax_kpc', 'Mgas'])

# Step 1: Core Baryonic Parameters
galaxies['M_tot_kg'] = (galaxies['Mgas'] + galaxies['MstarSED']) * 1e7 * M_sun
galaxies['Gas_Fraction'] = galaxies['Mgas'] / (galaxies['Mgas'] + galaxies['MstarSED'])
galaxies['R_s_meters'] = (2 * G * galaxies['M_tot_kg']) / (c**2)

# Step 2: Global Limits & Asymptotic Speeds
galaxies['R_trans_meters'] = np.sqrt(1.5 * np.pi * galaxies['R_s_meters'] * R_H)
galaxies['R_trans_kpc'] = galaxies['R_trans_meters'] / kpc_to_m
galaxies['beta_0'] = (galaxies['R_s_meters'] / (6 * np.pi * R_H))**0.25
galaxies['V_flat_km_s'] = (galaxies['beta_0'] * c) / 1000

# Step 3: Finite-Radius Point-Mass Prediction
galaxies['e_0_edge'] = galaxies['Rmax_kpc'] / galaxies['R_trans_kpc']
galaxies['V_pred_finite_km_s'] = galaxies['V_flat_km_s'] * np.sqrt((1 + galaxies['e_0_edge']) / galaxies['e_0_edge'])

# Step 4: Residuals
galaxies['Error_Finite'] = galaxies['V_pred_finite_km_s'] - galaxies['V_obs_max_km_s']
galaxies['Abs_Error_Finite'] = galaxies['Error_Finite'].abs()

# Step 5: Filter clean subset (Gas_Fraction > 0.8, e_0_edge > 1.5)
clean_sample = galaxies[(galaxies['Gas_Fraction'] > 0.8) & (galaxies['e_0_edge'] > 1.5)].copy()

# Output Generation
med_ae = clean_sample['Abs_Error_Finite'].median()
med_bias = clean_sample['Error_Finite'].median()

print("WILL RG: LITTLE THINGS FINITE-RADIUS ANALYSIS")
print("=" * 85)
print(f"Total valid galaxies parsed: {len(galaxies)}")
print(f"Clean Subset (Gas Fraction > 0.8, e_0 > 1.5): {len(clean_sample)}")
print("-" * 85)
print("FINITE-RADIUS LIMIT (V_pred) METRICS:")
print(f"Median Absolute Error (MedAE): {med_ae:.2f} km/s")
print(f"Median Signed Bias (Theory - Obs): {med_bias:.2f} km/s")
print("-" * 85)

display_cols = ['Name', 'Gas_Fraction', 'e_0_edge', 'Rmax_kpc', 'V_obs_max_km_s', 'V_pred_finite_km_s', 'Error_Finite']
print("\nCLEAN SUBSET DETAILED BREAKDOWN:")
display(clean_sample[display_cols].round(3))

WILL RG: LITTLE THINGS FINITE-RADIUS ANALYSIS
Total valid galaxies parsed: 26
Clean Subset (Gas Fraction > 0.8, e_0 > 1.5): 18
-------------------------------------------------------------------------------------
FINITE-RADIUS LIMIT (V_pred) METRICS:
Median Absolute Error (MedAE): 8.34 km/s
Median Signed Bias (Theory - Obs): -3.49 km/s
-------------------------------------------------------------------------------------

CLEAN SUBSET DETAILED BREAKDOWN:


,Name,Gas_Fraction,e_0_edge,Rmax_kpc,V_obs_max_km_s,V_pred_finite_km_s,Error_Finite
8,CVnIdwA,0.856,9.972,2.59,23.5,24.890,1.390
9,DDO_43,1.000,6.168,4.19,33.2,41.369,8.169
10,DDO_46,1.000,4.412,2.92,66.1,41.953,-24.147
11,DDO_47,1.000,8.001,7.71,64.7,48.476,-16.224
12,DDO_50,0.926,5.822,9.81,35.7,65.425,29.725
13,DDO_52,0.863,6.194,5.43,60.2,46.983,-13.217
14,DDO_53,0.878,3.646,1.45,28.6,33.143,4.543
16,DDO_87,0.899,9.219,7.39,56.6,43.890,-12.710
18,DDO_126,0.910,6.680,3.99,37.2,38.582,1.382
19,DDO_133,0.809,6.198,3.48,46.1,37.597,-8.503
